# 03b: Joren et al. Labels (Colab CPU only)

Judge 1,500 questions with Joren et al. (ICLR 2025, App. C.1) sufficient-context prompt using Groq API.

**Test mode:** N_QUESTIONS = 20 (change to None for full 1,500).

In [1]:
# TEST RUN: set to 20. For full run, set to None.
N_QUESTIONS = 20

In [2]:
import os
import sys
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict
import re

import pandas as pd
import numpy as np
from scipy.stats import contingency

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running on Colab: {IN_COLAB}")

Running on Colab: True


In [3]:
# ── Setup: works on a laptop (VS Code) and on Colab ───────────────────────
import os, sys, json, subprocess
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    # Colab: get the code from GitHub (push your latest commit first) and mount Drive
    REPO, BRANCH = "/content/HALT-RAG", "lift_signal"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/Shaithra2107/HALT-RAG", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    # laptop: this notebook sits in <repo>/notebooks, so the repo is one level up
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))

from halt_rag.config import load_config, data_path, data_dir
cfg = load_config(os.path.join(REPO, "config.yaml"))
print("running on:", "Colab" if ON_COLAB else "laptop", "| data folder:", data_dir(cfg))

Mounted at /content/drive
running on: Colab | data folder: /content/drive/MyDrive/halt_rag_cache


In [4]:
# Groq config and paths from config.yaml
print(f"Groq limits: RPM={cfg['judge']['groq_rpm_limit']}, TPM={cfg['judge']['groq_tpm_limit']}, TPD={cfg['judge']['groq_tpd_limit']}")

# Reference config model and limits so we don't repeat the dict access everywhere
GROQ_MODEL = cfg['judge']['groq_model']
GROQ_TEMP = cfg['judge']['groq_temperature']
GROQ_MAX_TOKENS = cfg['judge']['groq_max_tokens']
GROQ_RPM = cfg['judge']['groq_rpm_limit']
GROQ_TPM = cfg['judge']['groq_tpm_limit']
GROQ_TPD = cfg['judge']['groq_tpd_limit']
GROQ_RETRY_WAIT = cfg['judge']['groq_retry_wait_secs']
GROQ_MAX_RETRIES = cfg['judge']['groq_max_retries']

Groq limits: RPM=30, TPM=15000, TPD=500000


In [5]:
# Read the Joren et al. prompt template
prompt_path = os.path.join(REPO, 'prompts', 'joren_sufficient_context.txt')
with open(prompt_path) as f:
    prompt_template = f.read()

print(f"Prompt loaded ({len(prompt_template)} chars)")
print(f"Template contains: <question>={('<question>' in prompt_template)}, <context>={('<context>' in prompt_template)}, <TIMESTAMP>={('<TIMESTAMP>' in prompt_template)}, TIMESTAMP={('TIMESTAMP' in prompt_template and 'TIMESTAMP' in prompt_template)}")

# Data paths from config.yaml
signals_path = data_path(cfg, "signals_1500")
checkpoint_path = data_path(cfg, "joren_1500_checkpoint")
output_path = data_path(cfg, "joren_1500")

print(f"Signals CSV: {signals_path}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Output: {output_path}")

Prompt loaded (2953 chars)
Template contains: <question>=True, <context>=True, <TIMESTAMP>=True, TIMESTAMP=True
Signals CSV: /content/drive/MyDrive/halt_rag_cache/halt_rag_1500_signals.csv
Checkpoint: /content/drive/MyDrive/halt_rag_cache/joren_1500_checkpoint
Output: /content/drive/MyDrive/halt_rag_cache/joren_1500


In [6]:
# Load halt_rag_1500_signals.csv
df = pd.read_csv(signals_path)

print(f"Loaded {len(df)} rows from {os.path.basename(signals_path)}")
print(f"Columns: {list(df.columns)}")

# Verify required columns
required_cols = ['id', 'question', 'band'] + [f'ret_passage_{i}' for i in range(1, 6)] + [f'ret_title_{i}' for i in range(1, 6)]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print(f"WARNING: Missing columns: {missing}")
    print(f"Available: {list(df.columns)}")
else:
    print(f"All required columns present")

# Check for y_cov (for agreement calculation)
has_y_cov = 'y_cov' in df.columns
print(f"Has y_cov: {has_y_cov}")

# Limit to N_QUESTIONS if specified
if N_QUESTIONS is not None:
    df = df.iloc[:N_QUESTIONS].copy()
    print(f"Limited to {len(df)} questions for test run")
else:
    print(f"Running on all {len(df)} questions")

Loaded 1500 rows from halt_rag_1500_signals.csv
Columns: ['question', 'answers_list', 'band', 'wiki_title', 's_uri', 's_pop', 'id', 'prop', 'qid', 'm2_status_w23', 'match_method_w23', 'lift', 'relevance', 'consistency', 'rand_mu', 'rand_sd', 'ret_idx', 'ret_passage_1', 'ret_title_1', 'ret_passage_2', 'ret_title_2', 'ret_passage_3', 'ret_title_3', 'ret_passage_4', 'ret_title_4', 'ret_passage_5', 'ret_title_5', 'kw_coverage', 'clarity', 'cas', 'cas_mean', 'cas_soft', 'answers', 'gen_ctx', 'y_gen', 'y_judge', 'y_cov', 'is_tail', 'evaluable', 'noise_ctrl']
All required columns present
Has y_cov: True
Limited to 20 questions for test run


In [7]:
# Initialize tracking
checkpoint_every = 25

results = []  # list of dicts: {id, question, band, y_joren, raw_output, parse_ok, model, timestamp}
parse_failures = []  # unparseable outputs for retry
sample_outputs = []  # first 3 successful outputs for printing

print(f"Checkpoint path: {checkpoint_path}")
print(f"Output path: {output_path}")

# Resume from checkpoint if it exists
done_ids = set()
if os.path.exists(checkpoint_path):
    checkpoint_df = pd.read_csv(checkpoint_path)
    done_ids = set(checkpoint_df['id'].values)
    print(f"Resuming: {len(done_ids)} questions already done")
    # Load the first 3 samples for display
    sample_outputs = checkpoint_df[checkpoint_df['parse_ok'] == True].head(3)['raw_output'].tolist()
else:
    print(f"Starting fresh (no checkpoint found)")

# Filter to questions not yet done
df_todo = df[~df['id'].isin(done_ids)].copy()
print(f"Questions to process: {len(df_todo)}")

Checkpoint path: /content/drive/MyDrive/halt_rag_cache/joren_1500_checkpoint
Output path: /content/drive/MyDrive/halt_rag_cache/joren_1500
Starting fresh (no checkpoint found)
Questions to process: 20


In [8]:
# Initialize Groq client
try:
    from groq import Groq
except ImportError:
    print("Installing groq...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "groq", "-q"])
    from groq import Groq

# Get API key from Colab Secrets (never print)
if ON_COLAB:
    from google.colab import userdata
    groq_api_key = userdata.get('GROQ_API_KEY')
else:
    groq_api_key = os.environ.get('GROQ_API_KEY')
    if not groq_api_key:
        raise ValueError("GROQ_API_KEY not found. Set it in environment or Colab Secrets.")

client = Groq(api_key=groq_api_key)
print(f"Groq client initialized")

Installing groq...
Groq client initialized


In [10]:
# Resume from checkpoint if it exists
done_ids = set()
if os.path.exists(checkpoint_path):
    checkpoint_df = pd.read_csv(checkpoint_path)
    done_ids = set(checkpoint_df['id'].values)
    print(f"Resuming: {len(done_ids)} questions already done")
    # Load the first 3 samples for display
    sample_outputs = checkpoint_df[checkpoint_df['parse_ok'] == True].head(3)['raw_output'].tolist()
else:
    print(f"Starting fresh (no checkpoint found)")

# Filter to questions not yet done
df_todo = df[~df['id'].isin(done_ids)].copy()
print(f"Questions to process: {len(df_todo)}")

Starting fresh (no checkpoint found)
Questions to process: 20


In [11]:
def estimate_tokens(text):
    """
    Rough estimate: 1 token ≈ 4 chars (Groq's tokenizer may differ, but this is conservative)
    """
    return len(text) // 4 + 1

def call_groq(client, prompt, model, temperature, max_tokens, rpm_limit, tpm_limit, tpd_limit, retry_wait_secs, max_retries):
    """
    Call Groq API with rate limiting and error handling using OpenAI-style SDK.
    Returns: (response_text, success, error_msg, usage_info)
    - success=True: got a valid response
    - success=False, error contains 'daily': hit daily limit, stop
    - success=False, error contains 'retry': hit per-minute limit, will retry
    """
    prompt_tokens = estimate_tokens(prompt)
    output_tokens = max_tokens
    total_tokens = prompt_tokens + output_tokens

    # Simple rate limiting: wait if needed
    # In production, track tokens across all requests, but for now assume we start fresh
    tokens_per_minute_estimate = total_tokens
    min_wait_secs = max(total_tokens / (tpm_limit / 60), 60 / rpm_limit)

    retry_count = 0
    while retry_count < max_retries:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content, True, None, response.usage
        except Exception as e:
            error_str = str(e)
            if '429' in error_str:
                if 'daily' in error_str.lower() or 'quota' in error_str.lower():
                    return None, False, "daily limit reached", None
                else:
                    # Per-minute limit
                    wait_time = retry_wait_secs * (2 ** retry_count)  # exponential backoff
                    print(f"  429 (per-minute): waiting {wait_time}s before retry {retry_count + 1}/{max_retries}...")
                    time.sleep(wait_time)
                    retry_count += 1
                    continue
            else:
                return None, False, f"API error: {error_str[:100]}", None

    return None, False, f"max retries ({max_retries}) exceeded", None

print("estimate_tokens and call_groq functions defined")

estimate_tokens and call_groq functions defined


In [13]:
def format_prompt(template, question, passages, titles, timestamp_str):
    """
    Fill in the Joren prompt template.
    passages: list of 5 passage texts
    titles: list of 5 passage titles
    Returns: filled prompt, or None if validation fails
    """
    # Format context as "Title: ... / Text: ..."
    context_parts = []
    for title, passage in zip(titles, passages):
        context_parts.append(f"Title: {title} / Text: {passage}")
    context_str = "\n".join(context_parts)

    # Fill template
    prompt = template.replace('<question>', question)
    prompt = prompt.replace('<context>', context_str)
    prompt = prompt.replace('<TIMESTAMP>', timestamp_str)
    prompt = prompt.replace('TIMESTAMP', timestamp_str)  # any bare TIMESTAMP

    # Assert no unfilled placeholders
    if '<question>' in prompt or '<context>' in prompt or '<TIMESTAMP>' in prompt:
        return None, "Unfilled placeholder remains"
    if 'TIMESTAMP' in prompt and timestamp_str not in prompt:
        return None, "Unfilled bare TIMESTAMP"

    return prompt, None

def parse_sufficient_context(raw_output):
    """
    Parse {"Sufficient Context": 0 or 1} from raw output.
    Returns: (label, success)
    - success=True: found and parsed
    - success=False: could not parse
    """
    # Try to find JSON-like structure
    # Look for "Sufficient Context": with value 0 or 1
    match = re.search(r'"Sufficient Context"\s*:\s*([01])', raw_output)
    if match:
        return int(match.group(1)), True

    # Try to parse as JSON
    try:
        # Find JSON object in output
        import ast
        obj = json.loads(raw_output)
        if 'Sufficient Context' in obj:
            val = obj['Sufficient Context']
            if val in [0, 1]:
                return val, True
    except:
        pass

    return None, False

def estimate_tokens(text):
    """
    Rough estimate: 1 token ≈ 4 chars (Groq's tokenizer may differ, but this is conservative)
    """
    return len(text) // 4 + 1

def call_groq(client, prompt, model, temperature, max_tokens, rpm_limit, tpm_limit, tpd_limit, retry_wait_secs, max_retries):
    """
    Call Groq API with rate limiting and error handling using OpenAI-style SDK.
    Returns: (response_text, success, error_msg, usage_info)
    - success=True: got a valid response
    - success=False, error contains 'daily': hit daily limit, stop
    - success=False, error contains 'retry': hit per-minute limit, will retry
    """
    prompt_tokens = estimate_tokens(prompt)
    output_tokens = max_tokens
    total_tokens = prompt_tokens + output_tokens

    # Simple rate limiting: wait if needed
    # In production, track tokens across all requests, but for now assume we start fresh
    tokens_per_minute_estimate = total_tokens
    min_wait_secs = max(total_tokens / (tpm_limit / 60), 60 / rpm_limit)

    retry_count = 0
    while retry_count < max_retries:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content, True, None, response.usage
        except Exception as e:
            error_str = str(e)
            if '429' in error_str:
                if 'daily' in error_str.lower() or 'quota' in error_str.lower():
                    return None, False, "daily limit reached", None
                else:
                    # Per-minute limit
                    wait_time = retry_wait_secs * (2 ** retry_count)  # exponential backoff
                    print(f"  429 (per-minute): waiting {wait_time}s before retry {retry_count + 1}/{max_retries}...")
                    time.sleep(wait_time)
                    retry_count += 1
                    continue
            else:
                return None, False, f"API error: {error_str[:100]}", None

    return None, False, f"max retries ({max_retries}) exceeded", None

print("Functions defined: format_prompt, parse_sufficient_context, estimate_tokens, call_groq")

Functions defined: format_prompt, parse_sufficient_context, estimate_tokens, call_groq


In [14]:
# Main loop
print(f"\nProcessing {len(df_todo)} questions...\n")

tokens_used_today = 0  # rough tracking
daily_limit_hit = False
timestamp_str = '2023-11-01'
api_error_ids = []  # track questions that hit API errors (to retry later)

for idx, (_, row) in enumerate(df_todo.iterrows(), 1):
    q_id = row['id']
    question = row['question']
    band = row['band']

    # Get 5 passages and titles
    passages = [row.get(f'ret_passage_{i}', '') for i in range(1, 6)]
    titles = [row.get(f'ret_title_{i}', '') for i in range(1, 6)]

    # Format prompt
    prompt, format_err = format_prompt(prompt_template, question, passages, titles, timestamp_str)
    if prompt is None:
        print(f"[{idx}/{len(df_todo)}] ID {q_id}: format error: {format_err}")
        results.append({
            'id': q_id,
            'question': question,
            'band': band,
            'y_joren': None,
            'raw_output': '',
            'parse_ok': False,
            'model': GROQ_MODEL,
            'timestamp': datetime.now(timezone.utc).isoformat(),
        })
        continue

    # Call Groq
    response, success, error, usage = call_groq(
        client, prompt,
        model=GROQ_MODEL,
        temperature=GROQ_TEMP,
        max_tokens=GROQ_MAX_TOKENS,
        rpm_limit=GROQ_RPM,
        tpm_limit=GROQ_TPM,
        tpd_limit=GROQ_TPD,
        retry_wait_secs=GROQ_RETRY_WAIT,
        max_retries=GROQ_MAX_RETRIES,
    )

    if not success:
        if 'daily' in error:
            print(f"\n*** DAILY LIMIT REACHED ***")
            print(f"Questions processed: {len([r for r in results if r['y_joren'] is not None or r['parse_ok']])}")
            print(f"Resume by running this notebook again (checkpoint saved).")
            daily_limit_hit = True
            break
        else:
            print(f"[{idx}/{len(df_todo)}] ID {q_id}: API error: {error}")
            # Track API error for retry, but do NOT save to results yet
            api_error_ids.append((q_id, question, band))
            continue

    # Parse response
    label, parse_ok = parse_sufficient_context(response)
    if not parse_ok:
        print(f"[{idx}/{len(df_todo)}] ID {q_id}: unparseable (will retry once)")
        parse_failures.append((q_id, question, band, response))

    # Store result (only if we got a real response, parsed or not)
    results.append({
        'id': q_id,
        'question': question,
        'band': band,
        'y_joren': label if parse_ok else None,
        'raw_output': response,
        'parse_ok': parse_ok,
        'model': GROQ_MODEL,
        'timestamp': datetime.now(timezone.utc).isoformat(),
    })

    if parse_ok and len(sample_outputs) < 3:
        sample_outputs.append(response)

    if idx % 10 == 0:
        print(f"[{idx}/{len(df_todo)}] Processed {idx} questions, {sum(1 for r in results if r['parse_ok'])} parsed")

    # Save checkpoint every 25 questions (only rows with real responses, not API errors)
    if idx % checkpoint_every == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv(checkpoint_path, index=False)
        print(f"  Checkpoint saved: {len(checkpoint_df)} rows with responses")

print(f"\nLoop finished. Total with responses: {len(results)}")
if api_error_ids:
    print(f"API errors encountered: {len(api_error_ids)} (will retry)")


Processing 20 questions...

[1/20] ID 4804637: API error: API error: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you d
[2/20] ID 4870620: API error: API error: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you d
[3/20] ID 5035834: API error: API error: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you d
[4/20] ID 5188636: API error: API error: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you d
[5/20] ID 5708121: API error: API error: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you d
[6/20] ID 2943987: API error: API error: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you d
[7/20] ID 5436371: API error: API error: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` d

In [15]:
# Save final output
results_df = pd.DataFrame(results)
results_df.to_csv(output_path, index=False)
print(f"Saved {len(results_df)} results to {os.path.basename(output_path)}")

# Also save checkpoint for next resume
results_df.to_csv(checkpoint_path, index=False)
print(f"Checkpoint updated")

Saved 0 results to joren_1500
Checkpoint updated


In [16]:
# Compute agreement with y_cov (if available)
if has_y_cov:
    # Merge results with y_cov
    merged = results_df.merge(df[['id', 'y_cov']], on='id', how='left')

    # Filter to rows with both labels and parse_ok=True
    valid = merged[(merged['y_joren'].notna()) & (merged['y_cov'].notna()) & (merged['parse_ok'] == True)].copy()

    if len(valid) > 0:
        # 2x2 contingency table
        from sklearn.metrics import confusion_matrix, cohen_kappa_score

        cm = confusion_matrix(valid['y_cov'], valid['y_joren'], labels=[0, 1])
        accuracy = (valid['y_cov'] == valid['y_joren']).sum() / len(valid)
        kappa = cohen_kappa_score(valid['y_cov'], valid['y_joren'])

        print("\n" + "="*60)
        print("Agreement: y_joren vs y_cov")
        print("="*60)
        print(f"\n2×2 Contingency Table (y_cov rows × y_joren cols):")
        print(f"              y_joren=0  y_joren=1")
        print(f"y_cov=0       {cm[0, 0]:6d}     {cm[0, 1]:6d}")
        print(f"y_cov=1       {cm[1, 0]:6d}     {cm[1, 1]:6d}")
        print(f"\nOverall:")
        print(f"  Agreement: {accuracy:.1%}")
        print(f"  Cohen's κ: {kappa:.3f}")
        print(f"  N: {len(valid)}")

        # By band
        for band_val in ['head', 'tail']:
            band_valid = valid[valid['band'] == band_val]
            if len(band_valid) > 0:
                acc = (band_valid['y_cov'] == band_valid['y_joren']).sum() / len(band_valid)
                kappa_band = cohen_kappa_score(band_valid['y_cov'], band_valid['y_joren'])
                print(f"\n{band_val.upper()}:")
                print(f"  Agreement: {acc:.1%}")
                print(f"  Cohen's κ: {kappa_band:.3f}")
                print(f"  N: {len(band_valid)}")
    else:
        print("No valid rows for agreement calculation (need both y_cov and y_joren parsed)")
else:
    print("y_cov not available, skipping agreement calculation")

KeyError: 'id'

In [ ]:
# Print stats
parsed_count = results_df['parse_ok'].sum()
parse_rate = parsed_count / len(results_df) if len(results_df) > 0 else 0

print("\n" + "="*60)
print("Labeling Stats")
print("="*60)
print(f"Total questions: {len(results_df)}")
print(f"Successfully parsed: {parsed_count} ({parse_rate:.1%})")
print(f"Unparseable: {len(results_df) - parsed_count}")

if parsed_count > 0:
    parsed_df = results_df[results_df['parse_ok'] == True]
    pos_count = (parsed_df['y_joren'] == 1).sum()
    neg_count = (parsed_df['y_joren'] == 0).sum()
    print(f"\nLabels (parsed only):")
    print(f"  Sufficient Context = 1: {pos_count} ({pos_count/len(parsed_df):.1%})")
    print(f"  Sufficient Context = 0: {neg_count} ({neg_count/len(parsed_df):.1%})")

    # By band
    for band_val in ['head', 'tail']:
        band_df = parsed_df[parsed_df['band'] == band_val]
        if len(band_df) > 0:
            band_pos = (band_df['y_joren'] == 1).sum()
            print(f"\n{band_val.upper():5s}: sufficient={band_pos}/{len(band_df)} ({band_pos/len(band_df):.1%})")

In [ ]:
print("\nDone.")
if daily_limit_hit:
    print("Daily quota hit. Checkpoint saved. Run this notebook again to resume.")
else:
    print(f"All questions processed. Output: {output_path.name}")